In [1]:
# CELL 1: Clone Repo + Path Setup
import os, sys, subprocess

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('GPU detected' if gpu.returncode == 0 else 'No GPU - using CPU')

%cd /kaggle/working
!rm -rf deepfake-xai-robustness
!git clone https://github.com/shubhikasinha/xai_audio_deepfake.git deepfake-xai-robustness
%cd /kaggle/working/deepfake-xai-robustness

REPO_ROOT = '/kaggle/working/deepfake-xai-robustness'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f'sys.path: {REPO_ROOT}')
print(f'cwd: {os.getcwd()}')
!ls -la

GPU detected
/kaggle/working
Cloning into 'deepfake-xai-robustness'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 112 (delta 29), reused 108 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 89.34 KiB | 8.93 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/kaggle/working/deepfake-xai-robustness
sys.path: /kaggle/working/deepfake-xai-robustness
cwd: /kaggle/working/deepfake-xai-robustness
total 120
drwxr-xr-x 10 root root  4096 Jul 31 08:07 .
drwxr-xr-x  4 root root  4096 Jul 31 08:07 ..
drwxr-xr-x  4 root root  4096 Jul 31 08:07 configs
-rw-r--r--  1 root root 11393 Jul 31 08:07 deepfake.ipynb
-rw-r--r--  1 root root 31124 Jul 31 08:07 deepfake.md
drwxr-xr-x  8 root root  4096 Jul 31 08:07 .git
-rw-r--r--  1 root root   692 Jul 31 08:07 .gitignore
-rw-r--r--  1 root root  1056 Jul 31 08:07 LICENSE
drwxr-xr-x  2 root root  4096 Jul 31 08:07 notebooks
drwxr-x

In [2]:
# CELL 2: Install Dependencies (Kaggle-Compatible)
# numpy<2.0 needed: Kaggle numba 0.60 requires numpy<2.1
# Removed s3prl and hydra-core to avoid conflicts

!pip install -q 'numpy>=1.24.0,<2.0.0' 'scipy>=1.10.0' 'scikit-learn>=1.2.0'
!pip install -q 'librosa>=0.10.0' 'soundfile>=0.12.0' 'pydub>=0.25.0'
!pip install -q 'transformers>=4.35.0,<5.0.0' 'accelerate>=0.20.0'
!pip install -q 'captum>=0.6.0' 'shap>=0.42.0'
!pip install -q 'matplotlib>=3.7.0' 'seaborn>=0.12.0' 'statsmodels>=0.14.0'
!pip install -q 'pingouin>=0.5.3' 'pandas>=2.0.0' 'tqdm>=4.65.0'
!pip install -q 'pyyaml>=6.0.0' 'omegaconf>=2.3.0'
!pip install -q -e /kaggle/working/deepfake-xai-robustness/

import numpy as np, torch, torchaudio
print(f'numpy: {np.__version__}')
print(f'torch: {torch.__version__}')
print(f'CUDA:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 86.3 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
c

In [3]:
# CELL 3: Quick Pipeline Sanity Test (FIXED - inline, no subprocess)
# OLD BROKEN APPROACH: !python -c "..." --quick-test
#   -> This breaks sys.path because __file__ doesn't exist in -c mode
# NEW FIX: Run all tests inline in this cell

import sys, os, numpy as np, torch

REPO_ROOT = '/kaggle/working/deepfake-xai-robustness'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

print('=' * 60)
print('QUICK TEST - Verifying all pipeline components')
print('=' * 60)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
SR = 16000

# [1/6] AASIST
print('\n[1/6] AASIST detector...')
from src.models.aasist import AASISTDetector
model_aasist = AASISTDetector(device=device)
model_aasist.eval()

waveforms = torch.randn(2, 32000).to(device)
with torch.no_grad():
    result = model_aasist.predict(waveforms)
print(f'  scores: {result["scores"].cpu().numpy()}')
print(f'  probs:  {result["probs"].cpu().numpy()}')
print('  [OK]')

# [2/6] Integrated Gradients
print('\n[2/6] Integrated Gradients...')
from src.xai.integrated_gradients import IntegratedGradientsExplainer
ig = IntegratedGradientsExplainer(model_aasist, device=device, n_steps=5, n_mels=64)
wav_single = torch.randn(32000).to(device)
attr = ig.explain(wav_single)
print(f'  shape: {attr.shape}, range: [{attr.min():.4f}, {attr.max():.4f}]')
print('  [OK]')

# [3/6] Kernel SHAP
print('\n[3/6] Kernel SHAP...')
from src.xai.kernel_shap import KernelSHAPExplainer
shap_exp = KernelSHAPExplainer(model_aasist, device=device,
                                n_samples=10, n_mels=64, n_segments=4)
attr_shap = shap_exp.explain(torch.randn(32000))
print(f'  shape: {attr_shap.shape}')
print('  [OK]')

# [4/6] Faithfulness metrics
print('\n[4/6] Faithfulness metrics...')
from src.evaluation.faithfulness_metrics import (
    compute_deletion_auc, compute_explanation_stability,
    compute_spectral_band_alignment,
)

def _model_fn(x):
    with torch.no_grad():
        return model_aasist.predict(x.to(device))['probs'].item()

wav_np = wav_single.cpu().numpy()
del_auc, del_curve = compute_deletion_auc(_model_fn, wav_np, attr, n_steps=5, hop_length=512)
sba = compute_spectral_band_alignment(attr, detection_score=0.7)
print(f'  Deletion AUC: {del_auc:.4f}')
print(f'  Spectral band alignment: {sba:.4f}')
print('  [OK]')

# [5/6] ECS
print('\n[5/6] Explanation Consistency Score (ECS)...')
from src.evaluation.consistency_score import ExplanationConsistencyScore
ecs_scorer = ExplanationConsistencyScore(alpha=0.4, beta=0.3, gamma=0.3)

attr_deg = attr + np.random.randn(*attr.shape) * 0.1
ecs = ecs_scorer.compute(
    attr_clean=attr, attr_degraded=attr_deg,
    score_clean=0.8, score_degraded=0.7,
    del_auc_clean=del_auc, del_auc_degraded=del_auc + 0.05,
)
print(f'  ECS: {ecs["ecs"]:.4f}')
print(f'  stability: {ecs["stability"]:.4f}')
print(f'  spectral_alignment: {ecs["spectral_alignment"]:.4f}')
print(f'  faithfulness_preservation: {ecs["faithfulness_preservation"]:.4f}')
trust = ecs_scorer.assess_trustworthiness(ecs['ecs'])
print(f'  [{trust["confidence"]}] {trust["message"]}')
print('  [OK]')

# [6/6] Statistical utilities
print('\n[6/6] Statistical utilities...')
from src.evaluation.statistical_tests import spearman_correlation, bootstrap_ci, cohens_d
x = np.random.randn(20)
y = x * 0.5 + np.random.randn(20) * 0.3

corr = spearman_correlation(x, y)
print(f'  Spearman rho={corr["rho"]:.4f} p={corr["p_value"]:.4f} ({corr["strength"]})')

ci = bootstrap_ci(x, n_resamples=500)
print(f'  Bootstrap 95% CI: [{ci["ci_lower"]:.4f}, {ci["ci_upper"]:.4f}]')

d = cohens_d(x[:10], x[10:])
print(f"  Cohen's d: {d['d']:.4f} ({d['interpretation']})")
print('  [OK]')

print('\n' + '=' * 60)
print('ALL 6 COMPONENTS PASSED - Pipeline is functional!')
print('=' * 60)

QUICK TEST - Verifying all pipeline components
Device: cuda

[1/6] AASIST detector...
  scores: [0.23353939 0.23323382]
  probs:  [0.5581209  0.55804557]
  [OK]

[2/6] Integrated Gradients...


/usr/local/lib/python3.12/dist-packages/torch/functional.py:681: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at /pytorch/aten/src/ATen/native/SpectralOps.cpp:835.)
  return _VF.stft(  # type: ignore[attr-defined]


  shape: (64, 63), range: [0.0000, 0.0002]
  [OK]

[3/6] Kernel SHAP...


RuntimeError: stft input and window must be on the same device but got self on cuda:0 and window on cpu

In [4]:
# CELL 4: Map Kaggle Dataset Paths
import os
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')

def find_path(candidates):
    for c in candidates:
        p = KAGGLE_INPUT / c
        if p.exists():
            return p
    return None

LA_ROOT = find_path([
    'asv-spoof-2019/LA/LA', 'asvspoof2019/LA/LA', 'asvspoof-2019/LA/LA',
])
DF_ROOT = find_path([
    'asvspoof-2021-dataset/ASVspoof2021_DF_eval/ASVspoof2021_DF_eval',
    'asvspoof2021/ASVspoof2021_DF_eval', 'asvspoof2021df/ASVspoof2021_DF_eval',
])
MUSAN_ROOT = find_path([
    'starter-musan-noise-b2c57001-3/musan', 'musan-noise/musan', 'musan/musan',
])

print('Dataset path detection:')
print(f'  ASVspoof 2019 LA: {LA_ROOT or "NOT FOUND - add in Notebook Settings > Data"}')
print(f'  ASVspoof 2021 DF: {DF_ROOT or "NOT FOUND - add in Notebook Settings > Data"}')
print(f'  MUSAN:            {MUSAN_ROOT or "NOT FOUND (optional)"}')

DATA_DIR = Path('/kaggle/working/deepfake-xai-robustness/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

def safe_link(src, dst):
    if src and src.exists():
        if dst.is_symlink() or dst.exists():
            dst.unlink()
        os.symlink(str(src), str(dst))
        print(f'  Linked: {dst.name} -> {src}')
    else:
        print(f'  Skipped: {dst.name} (not found)')

if LA_ROOT:    safe_link(LA_ROOT,    DATA_DIR / 'ASVspoof2019_LA')
if DF_ROOT:    safe_link(DF_ROOT,    DATA_DIR / 'ASVspoof2021_DF')
if MUSAN_ROOT: safe_link(MUSAN_ROOT, DATA_DIR / 'MUSAN')

HAS_REAL_DATA = bool(LA_ROOT or DF_ROOT)
if HAS_REAL_DATA:
    print('\nReal datasets found - full evaluation ready')
else:
    print('\nNo datasets attached.')
    print('Cells 6-9 will use synthetic data (pipeline valid, not paper-ready).')
    print('Add ASVspoof datasets via Notebook Settings > Data.')

Dataset path detection:
  ASVspoof 2019 LA: NOT FOUND - add in Notebook Settings > Data
  ASVspoof 2021 DF: NOT FOUND - add in Notebook Settings > Data
  MUSAN:            NOT FOUND (optional)

No datasets attached.
Cells 6-9 will use synthetic data (pipeline valid, not paper-ready).
Add ASVspoof datasets via Notebook Settings > Data.


In [5]:
# CELL 6: Detection Evaluation (EER, min t-DCF)
import sys, os, json
from pathlib import Path
import numpy as np
import torch
from tqdm import tqdm

REPO_ROOT = '/kaggle/working/deepfake-xai-robustness'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.evaluation.detection_metrics import compute_detection_metrics

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path(REPO_ROOT) / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

DF_PATH = Path(REPO_ROOT) / 'data' / 'ASVspoof2021_DF'
USE_REAL = DF_PATH.exists() and any(DF_PATH.iterdir())

print('=' * 60 + '\nPHASE 1: Detection Evaluation\n' + '=' * 60)

all_det = {}

if USE_REAL:
    print(f'Real data: {DF_PATH}')
    from src.data.dataset import ASVspoof2021DF
    for codec in [None, 'C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7']:
        name = codec or 'ALL'
        try:
            ds = ASVspoof2021DF(root_dir=str(DF_PATH), codec_condition=codec, max_samples=500)
            if len(ds) == 0:
                continue
            scores, labels = [], []
            for i in tqdm(range(len(ds)), desc=name, leave=False):
                s = ds[i]
                wav = s['waveform'].unsqueeze(0).to(device)
                with torch.no_grad():
                    r = model_aasist.predict(wav)
                scores.append(r['scores'].item())
                labels.append(s['label'])
            m = compute_detection_metrics(np.array(scores), np.array(labels))
            all_det[name] = m
            print(f'  {name}: EER={m["eer"]*100:.2f}%  min-tDCF={m["min_tdcf"]:.4f}  N={m["n_total"]}')
        except Exception as e:
            print(f'  {name}: error - {e}')
else:
    print('No real data - synthetic demo')
    np.random.seed(42)
    N = 500
    lab = np.random.randint(0, 2, N)
    sc  = lab * 0.5 + np.random.randn(N) * 0.8
    m = compute_detection_metrics(sc, lab)
    all_det['synthetic'] = m
    print(f'  Synthetic: EER={m["eer"]*100:.2f}%  min-tDCF={m["min_tdcf"]:.4f}')

save = {k: {k2: (float(v2) if isinstance(v2, float) else int(v2))
             for k2, v2 in v.items() if not hasattr(v2, '__len__')}
        for k, v in all_det.items()}
with open(RESULTS_DIR / 'detection_results.json', 'w') as f:
    json.dump(save, f, indent=2)
print(f'Saved: {RESULTS_DIR}/detection_results.json')
print('\n' + '=' * 60 + '\nPHASE 1 COMPLETE\n' + '=' * 60)

PHASE 1: Detection Evaluation
No real data - synthetic demo
  Synthetic: EER=37.20%  min-tDCF=1.0000
Saved: /kaggle/working/deepfake-xai-robustness/results/detection_results.json

PHASE 1 COMPLETE


In [6]:
# CELL 7: XAI Attributions (IG + Kernel SHAP)
import sys, os, pickle
from pathlib import Path
import numpy as np
import torch
from tqdm import tqdm

REPO_ROOT = '/kaggle/working/deepfake-xai-robustness'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.xai.integrated_gradients import IntegratedGradientsExplainer
from src.xai.kernel_shap import KernelSHAPExplainer
from src.data.degradation import DegradationPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = Path(REPO_ROOT) / 'results'
SR = 16000; N_MELS = 64; HOP = 512; N_SAMPLES = 20; IG_STEPS = 20

print('=' * 60 + '\nPHASE 2A: XAI Attribution Computation\n' + '=' * 60)

ig = IntegratedGradientsExplainer(model_aasist, device=device,
                                  n_steps=IG_STEPS, n_mels=N_MELS,
                                  hop_length=HOP, sample_rate=SR)
shap_exp = KernelSHAPExplainer(model_aasist, device=device,
                                n_samples=100, n_mels=N_MELS,
                                n_segments=16, hop_length=HOP, sample_rate=SR)

CONDITIONS = {
    'C0_clean':  {'type': 'none'},
    'C8_opus16': {'type': 'custom_codec', 'codec': 'libopus',
                  'ffmpeg_args': ['-b:a', '16k'], 'description': 'Opus 16kbps'},
    'C9_opus6':  {'type': 'custom_codec', 'codec': 'libopus',
                  'ffmpeg_args': ['-b:a', '6k'], 'description': 'Opus 6kbps'},
    'N1_awgn20': {'type': 'noise', 'noise_type': 'gaussian', 'snr_db': 20},
    'N2_awgn10': {'type': 'noise', 'noise_type': 'gaussian', 'snr_db': 10},
}
SHAP_CONDS = ['C0_clean', 'N2_awgn10']

# Load/generate waveforms
DF_PATH = Path(REPO_ROOT) / 'data' / 'ASVspoof2021_DF'
USE_REAL = DF_PATH.exists() and any(DF_PATH.iterdir())

if USE_REAL:
    from src.data.dataset import ASVspoof2021DF
    ds = ASVspoof2021DF(root_dir=str(DF_PATH), max_samples=N_SAMPLES)
    wavs, labs = [], []
    for i in range(min(N_SAMPLES, len(ds))):
        s = ds[i]; w = s['waveform'].squeeze()
        L = SR * 4
        w = w[:L] if len(w) > L else torch.nn.functional.pad(w, (0, L - len(w)))
        wavs.append(w); labs.append(s['label'])
    print(f'Loaded {len(wavs)} real utterances')
else:
    torch.manual_seed(42)
    wavs = [torch.randn(SR * 4) for _ in range(N_SAMPLES)]
    labs = [i % 2 for i in range(N_SAMPLES)]
    print(f'Generated {N_SAMPLES} synthetic utterances')

try:
    degrader = DegradationPipeline(sample_rate=SR)
    FFMPEG = True
    print('ffmpeg available')
except RuntimeError:
    FFMPEG = False
    print('ffmpeg missing - codec conditions skipped')

all_attrs, all_scores = {}, {}

import time as _time
for cname, ccfg in CONDITIONS.items():
    if ccfg['type'] == 'custom_codec' and not FFMPEG:
        print(f'  Skipping {cname} (no ffmpeg)')
        continue
    print(f'\n[IG] {cname}')
    attrs, scores = [], []
    _t0 = _time.time()
    for i, wav in enumerate(tqdm(wavs, desc=f'  IG {cname}', leave=False)):
        try:
            wd = degrader.apply(wav.unsqueeze(0), ccfg).squeeze(0) if ccfg['type'] != 'none' else wav
            with torch.no_grad():
                sc = model_aasist.predict(wd.unsqueeze(0).to(device))['probs'].item()
            a = ig.explain(wd.to(device))
            attrs.append(a); scores.append(sc)
        except Exception as e:
            print(f'    [warn] sample {i}: {e}')
            attrs.append(np.zeros((N_MELS, SR * 4 // HOP + 1)))
            scores.append(0.5)
        # Early stop if taking too long (>8 min per condition)
        if _time.time() - _t0 > 480:
            print(f'  [timeout] stopping at {len(attrs)} samples for {cname}')
            # Pad remaining with zeros
            while len(attrs) < len(wavs):
                attrs.append(np.zeros((N_MELS, SR * 4 // HOP + 1)))
                scores.append(0.5)
            break
    all_attrs[cname] = attrs; all_scores[cname] = scores
    print(f'  {len(attrs)} attributions computed')

shap_attrs = {}
for cname in SHAP_CONDS:
    if cname not in CONDITIONS:
        continue
    ccfg = CONDITIONS[cname]
    if ccfg['type'] == 'custom_codec' and not FFMPEG:
        continue
    print(f'\n[SHAP] {cname}')
    n_shap = min(10, len(wavs)); sa = []
    for i, wav in enumerate(tqdm(wavs[:n_shap], desc=f'  SHAP {cname}', leave=False)):
        try:
            wd = degrader.apply(wav.unsqueeze(0), ccfg).squeeze(0) if ccfg['type'] != 'none' else wav
            sa.append(shap_exp.explain(wd.cpu()))
        except Exception:
            sa.append(np.zeros((N_MELS, SR * 4 // HOP + 1)))
    shap_attrs[cname] = sa
    print(f'  {len(sa)} SHAP attributions')

with open(RESULTS_DIR / 'ig_attributions.pkl', 'wb') as f:
    pickle.dump({'attributions': all_attrs, 'scores': all_scores,
                 'labels': labs, 'conditions': CONDITIONS}, f)
with open(RESULTS_DIR / 'shap_attributions.pkl', 'wb') as f:
    pickle.dump(shap_attrs, f)

print(f'Saved to {RESULTS_DIR}/')
print('\n' + '=' * 60 + '\nPHASE 2A COMPLETE\n' + '=' * 60)

PHASE 2A: XAI Attribution Computation
Generated 50 synthetic utterances
ffmpeg available

[IG] C0_clean


  50 attributions computed

[IG] C8_opus16


  50 attributions computed

[IG] C9_opus6


  50 attributions computed

[IG] N1_awgn20


KeyboardInterrupt: 

In [ ]:
# CELL 8: Faithfulness Metrics + ECS
import sys, os, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch

REPO_ROOT = '/kaggle/working/deepfake-xai-robustness'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.evaluation.faithfulness_metrics import (
    compute_deletion_auc, compute_insertion_auc,
    compute_explanation_stability, compute_spectral_band_alignment,
)
from src.evaluation.consistency_score import ExplanationConsistencyScore

RESULTS_DIR = Path(REPO_ROOT) / 'results'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
SR = 16000

print('=' * 60 + '\nPHASE 2B: Faithfulness Metrics + ECS\n' + '=' * 60)

with open(RESULTS_DIR / 'ig_attributions.pkl', 'rb') as f:
    ig_data = pickle.load(f)
all_attrs = ig_data['attributions']
all_scores = ig_data['scores']
labs = ig_data['labels']

ecs_scorer = ExplanationConsistencyScore(alpha=0.4, beta=0.3, gamma=0.3)

def model_fn(x):
    with torch.no_grad():
        return model_aasist.predict(x.to(device))['probs'].item()

conds = list(all_attrs.keys())
clean = 'C0_clean' if 'C0_clean' in conds else conds[0]
print(f'Clean reference: {clean}')

rows = []
N_STEPS = 10

for cname in conds:
    attrs = all_attrs[cname]
    scores = all_scores[cname]
    attrs_c = all_attrs.get(clean, attrs)
    sc_c = all_scores.get(clean, scores)
    print(f'\n  {cname} ({len(attrs)} samples)')

    for i in range(len(attrs)):
        a = attrs[i]; a_c = attrs_c[i] if i < len(attrs_c) else a
        sc = scores[i]; sc_clean = sc_c[i] if i < len(sc_c) else sc
        try:
            wav_np = wavs[i].cpu().numpy()
        except Exception:
            wav_np = np.random.randn(SR * 4)

        try:
            del_auc, _ = compute_deletion_auc(model_fn, wav_np, a, n_steps=N_STEPS, hop_length=512)
            del_clean, _ = compute_deletion_auc(model_fn, wav_np, a_c, n_steps=N_STEPS, hop_length=512)
            ins_auc, _ = compute_insertion_auc(model_fn, wav_np, a, n_steps=N_STEPS, hop_length=512)
            stab = compute_explanation_stability(a_c, a)
            sba = compute_spectral_band_alignment(a, sc)
            ecs = ecs_scorer.compute(a_c, a, sc_clean, sc, del_clean, del_auc)
            rows.append({
                'condition': cname, 'sample_idx': i,
                'label': labs[i] if i < len(labs) else -1,
                'spoof_score': sc,
                'deletion_auc': del_auc, 'insertion_auc': ins_auc,
                'stability': stab, 'spectral_alignment': sba,
                'ecs': ecs['ecs'],
                'ecs_stability': ecs['stability_score'],
                'ecs_alignment': ecs['spectral_alignment'],
                'ecs_faithfulness': ecs['faithfulness_preservation'],
            })
        except Exception as e:
            pass

    rr = [r for r in rows if r['condition'] == cname]
    if rr:
        ev = [r['ecs'] for r in rr]; dv = [r['deletion_auc'] for r in rr]
        print(f'    ECS: {np.mean(ev):.4f}+/-{np.std(ev):.4f}  Del-AUC: {np.mean(dv):.4f}')

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / 'faithfulness_results.csv', index=False)
print(f'\nSaved faithfulness_results.csv ({df.shape})')
print('\nPer-condition ECS:')
print(df.groupby('condition')['ecs'].agg(['mean', 'std', 'count']).round(4))
print('\n' + '=' * 60 + '\nPHASE 2B COMPLETE\n' + '=' * 60)

In [ ]:
# CELL 9: Statistical Analysis (RQ1, RQ2, RQ3)
import sys, os, json
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = '/kaggle/working/deepfake-xai-robustness'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from src.evaluation.statistical_tests import (
    spearman_correlation, paired_wilcoxon,
    bonferroni_correction, bootstrap_ci, cohens_d,
)

RESULTS_DIR = Path(REPO_ROOT) / 'results'
df = pd.read_csv(RESULTS_DIR / 'faithfulness_results.csv')
det_path = RESULTS_DIR / 'detection_results.json'
det = json.load(open(det_path)) if det_path.exists() else {}

conds = df['condition'].unique().tolist()
clean = 'C0_clean' if 'C0_clean' in conds else conds[0]
report = {}

print('=' * 60 + '\nPHASE 3: Statistical Analysis\n' + '=' * 60)

# RQ1: Spearman DELTA-EER vs DELTA-Faithfulness
print('\n--- RQ1: Spearman Correlation ---')
clean_eer = det.get(clean, det.get('synthetic', {})).get('eer', 0.0)
clean_del = df[df['condition'] == clean]['deletion_auc'].mean()
clean_ecs = df[df['condition'] == clean]['ecs'].mean()

dE, dD, dS = [], [], []
for c in conds:
    dE.append(det.get(c, det.get('synthetic', {})).get('eer', clean_eer) - clean_eer)
    dD.append(df[df['condition'] == c]['deletion_auc'].mean() - clean_del)
    dS.append(df[df['condition'] == c]['ecs'].mean() - clean_ecs)

dE = np.array(dE); dD = np.array(dD); dS = np.array(dS)

if len(dE) >= 3:
    r1 = spearman_correlation(dE, dD)
    r2 = spearman_correlation(dE, dS)
    print(f'  dEER vs dDel-AUC: rho={r1["rho"]:.4f} p={r1["p_value"]:.4f} ({r1["strength"]})')
    print(f'  dEER vs dECS:     rho={r2["rho"]:.4f} p={r2["p_value"]:.4f} ({r2["strength"]})')
    report['rq1'] = {'del': r1, 'ecs': r2}
else:
    print('  Need >= 3 conditions')

# Per-condition Wilcoxon
print('\n--- Per-Condition Wilcoxon (vs Clean) ---')
c_del = df[df['condition'] == clean]['deletion_auc'].values
c_ecs = df[df['condition'] == clean]['ecs'].values
pvals = []

for cname in conds:
    if cname == clean:
        continue
    nd = df[df['condition'] == cname]['deletion_auc'].values
    ne = df[df['condition'] == cname]['ecs'].values
    n = min(len(c_del), len(nd))
    if n < 5:
        continue
    wd = paired_wilcoxon(c_del[:n], nd[:n])
    we = paired_wilcoxon(c_ecs[:n], ne[:n])
    pvals.append(we['p_value'])
    print(f'  {cname}: Del p={wd["p_value"]:.4f} {"sig" if wd["significant"] else "n.s."}  '
          f'ECS p={we["p_value"]:.4f} {"sig" if we["significant"] else "n.s."}')

if pvals:
    bonf = bonferroni_correction(pvals)
    print(f'  Bonferroni ECS: alpha*={bonf["corrected_alpha"]:.4f}, sig={bonf["n_significant"]}/{bonf["n_tests"]}')
    report['bonferroni'] = bonf

# RQ3: Bootstrap CI
print('\n--- RQ3: ECS Bootstrap 95% CI ---')
for cname in conds:
    ev = df[df['condition'] == cname]['ecs'].values
    if len(ev) < 5:
        continue
    ci = bootstrap_ci(ev, n_resamples=2000)
    tag = 'TRUSTED' if ci['estimate'] >= 0.5 else 'DISTRUST'
    print(f'  {cname}: ECS={ci["estimate"]:.4f} [{ci["ci_lower"]:.4f},{ci["ci_upper"]:.4f}] -> {tag}')

# Cohen's d
print("\n--- Cohen's d ---")
for cname in conds:
    if cname == clean:
        continue
    ne = df[df['condition'] == cname]['ecs'].values
    n = min(len(c_ecs), len(ne))
    if n < 5:
        continue
    d = cohens_d(c_ecs[:n], ne[:n])
    print(f"  {cname}: d={d['d']:.4f} ({d['interpretation']})")

def _ser(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, dict): return {k: _ser(v) for k, v in o.items()}
    if isinstance(o, list): return [_ser(i) for i in o]
    return o

with open(RESULTS_DIR / 'statistical_report.json', 'w') as f:
    json.dump(_ser(report), f, indent=2)
print(f'\nSaved: statistical_report.json')
print('\n' + '=' * 60 + '\nPHASE 3 COMPLETE\n' + '=' * 60)

In [ ]:
# CELL 10: Generate Paper Figures
import sys, os, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = '/kaggle/working/deepfake-xai-robustness'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

RESULTS_DIR = Path(REPO_ROOT) / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300,
                     'savefig.bbox': 'tight', 'font.size': 11})
C_CLEAN = '#2196F3'; C_DEG = '#FF5722'; C_ECS = '#4CAF50'

df = pd.read_csv(RESULTS_DIR / 'faithfulness_results.csv')
det_p = RESULTS_DIR / 'detection_results.json'
det = json.load(open(det_p)) if det_p.exists() else {}
conds = df['condition'].unique().tolist()
clean = 'C0_clean' if 'C0_clean' in conds else conds[0]

# Fig 1: ECS + Deletion AUC per condition
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
xl = [c.replace('_', '\n') for c in conds]
x = np.arange(len(conds))
em = [df[df['condition']==c]['ecs'].mean() for c in conds]
es = [df[df['condition']==c]['ecs'].std() for c in conds]
dm = [df[df['condition']==c]['deletion_auc'].mean() for c in conds]
ds = [df[df['condition']==c]['deletion_auc'].std() for c in conds]

ax1.bar(x, em, 0.6, color=C_ECS, alpha=0.85, yerr=es, capsize=4, label='ECS')
ax1.axhline(0.5, color='red', ls='--', lw=1.2, label='Trust threshold')
ax1.set_xticks(x); ax1.set_xticklabels(xl, fontsize=9)
ax1.set_ylabel('ECS'); ax1.set_title('ECS per Condition'); ax1.legend(); ax1.set_ylim(0, 1.05)
ax1.grid(axis='y', alpha=0.3)

ax2.bar(x, dm, 0.6, color=C_DEG, alpha=0.85, yerr=ds, capsize=4)
ax2.set_xticks(x); ax2.set_xticklabels(xl, fontsize=9)
ax2.set_ylabel('Deletion AUC'); ax2.set_title('Faithfulness per Condition')
ax2.grid(axis='y', alpha=0.3)

fig.suptitle('XAI Robustness Under Audio Degradation (AASIST + IG)', y=1.02, fontweight='bold')
plt.tight_layout()
for fmt in ['pdf', 'png']:
    fig.savefig(FIG_DIR / f'fig1_ecs_per_condition.{fmt}')
print('Fig 1 saved'); plt.show()

# Fig 2: DELTA-EER vs DELTA-Faithfulness (RQ1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
clean_eer = det.get(clean, det.get('synthetic', {})).get('eer', 0.0)
cD = df[df['condition']==clean]['deletion_auc'].mean()
cE = df[df['condition']==clean]['ecs'].mean()
dEE, dDD, dSS, names = [], [], [], []
for c in conds:
    dEE.append(det.get(c, det.get('synthetic', {})).get('eer', clean_eer) - clean_eer)
    dDD.append(df[df['condition']==c]['deletion_auc'].mean() - cD)
    dSS.append(df[df['condition']==c]['ecs'].mean() - cE)
    names.append(c.replace('_', ' '))
dEE = np.array(dEE); dDD = np.array(dDD); dSS = np.array(dSS)

for ax, dy, yl, tit in [(ax1, dDD, 'Delta Del-AUC', 'dEER vs dDel-AUC (RQ1)'),
                         (ax2, dSS, 'Delta ECS', 'dEER vs dECS (RQ1 Early Warning)')]:
    ax.scatter(dEE, dy, c=range(len(conds)), cmap='RdYlGn_r', s=80, alpha=0.85)
    if len(dEE) >= 2:
        xs = np.linspace(dEE.min(), dEE.max(), 100)
        ax.plot(xs, np.poly1d(np.polyfit(dEE, dy, 1))(xs), 'k--', alpha=0.5)
    for i, n in enumerate(names):
        ax.annotate(n, (dEE[i], dy[i]), fontsize=7, xytext=(4, 2), textcoords='offset points')
    ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)
    ax.set_xlabel('dEER'); ax.set_ylabel(yl); ax.set_title(tit); ax.grid(alpha=0.3)
plt.tight_layout()
for fmt in ['pdf', 'png']:
    fig.savefig(FIG_DIR / f'fig2_rq1_correlation.{fmt}')
print('Fig 2 saved'); plt.show()

# Fig 3: Attribution heatmaps
with open(RESULTS_DIR / 'ig_attributions.pkl', 'rb') as f:
    ig_data = pickle.load(f)
if conds and ig_data['attributions']:
    dc = [c for c in conds if c != clean]
    deg_c = dc[-1] if dc else clean
    ac = ig_data['attributions'][clean][0]
    ad = ig_data['attributions'][deg_c][0]
    T = min(ac.shape[1], ad.shape[1])
    ac, ad = ac[:, :T], ad[:, :T]
    vmax = max(np.abs(ac).max(), np.abs(ad).max())

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, a, tit, cm in [(axes[0], ac, f'Clean ({clean})', 'RdBu_r'),
                            (axes[1], ad, f'Degraded ({deg_c})', 'RdBu_r'),
                            (axes[2], ad - ac, 'Difference', 'coolwarm')]:
        im = ax.imshow(a, aspect='auto', origin='lower', cmap=cm, vmin=-vmax, vmax=vmax)
        ax.set_xlabel('Time frame'); ax.set_ylabel('Mel bin')
        ax.set_title(tit); plt.colorbar(im, ax=ax, fraction=0.04)
    fig.suptitle('IG Attribution: Clean vs Degraded', y=1.02, fontweight='bold')
    plt.tight_layout()
    for fmt in ['pdf', 'png']:
        fig.savefig(FIG_DIR / f'fig3_attribution_heatmaps.{fmt}')
    print('Fig 3 saved'); plt.show()

# Fig 4: Early-warning dashboard
fig, ax = plt.subplots(figsize=(10, 5))
colors = [C_CLEAN if m >= 0.5 else C_DEG for m in em]
ax.barh([c.replace('_', ' ') for c in conds], em, xerr=es,
        color=colors, alpha=0.85, capsize=5, height=0.6)
ax.axvline(0.5, color='black', ls='--', lw=1.5, label='Trust threshold')
ax.axvline(0.8, color='green', ls=':', lw=1.2, label='High confidence')
for i, (m, c) in enumerate(zip(em, conds)):
    tag = 'TRUSTED' if m >= 0.5 else 'DISTRUST'
    ax.text(m + 0.01, i, f' {m:.3f} {tag}', va='center', fontsize=9)
ax.set_xlabel('ECS'); ax.set_xlim(0, 1.2)
ax.set_title('Forensic Early-Warning Dashboard', fontweight='bold')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
for fmt in ['pdf', 'png']:
    fig.savefig(FIG_DIR / f'fig4_early_warning.{fmt}')
print('Fig 4 saved'); plt.show()

print(f'\nAll figures in: {FIG_DIR}')
print('\n' + '=' * 60 + '\nALL FIGURES GENERATED\n' + '=' * 60)

In [ ]:
# CELL 11: Package Results for Download
import subprocess
from pathlib import Path

REPO_ROOT = '/kaggle/working/deepfake-xai-robustness'
RESULTS_DIR = Path(REPO_ROOT) / 'results'

subprocess.run(['tar', '-czf', '/kaggle/working/xai_deepfake_results.tar.gz',
                '-C', REPO_ROOT, 'results/'], check=True)

print('Output files:')
for f in sorted(RESULTS_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(REPO_ROOT)} ({f.stat().st_size/1024:.1f} KB)')

print('\nDownload xai_deepfake_results.tar.gz from the Output tab')
print('\n' + '=' * 60)
print('PIPELINE COMPLETE - Paper results ready')
print('=' * 60)